In [1]:
import os
import sys

import pandas as pd
import numpy as np
import anndata as ad

import tacco as tc
import scanpy as sc
import importlib

# The notebook expects to be executed either in the workflow directory or in the repository root folder...
#sys.path.insert(1, os.path.abspath('workflow' if os.path.exists('workflow/common_code.py') else '..'))
#import common_code

In [2]:
data_path = './xenium_ctnnd2_vc/data'

In [3]:
## Read in batch-corrected and annotated P28 vis ctx snRNAseq as reference data
snrnaseq_data_path = 'Z:/eroglulab_share/Gabi_Sejourne/Ctnnd2Zbtb20_Manuscript_DevCell_2025/Figure 3/snRNAseq_dataset/GEO_submission/processed_data/anndata_objects'
reference = ad.read_h5ad(f'{snrnaseq_data_path}/ctnnd2_scrnaseq_uncorrected_finalannot.h5ad')
print(reference.obs['cluster_id'].isna().sum())  # Should print 0

0


In [4]:
# Find indices with non-NaN cluster_id and ignore annotations added before scVI
valid_idx = reference.obs['cluster_id'].notna() & (reference.obs['cluster_id'] != "UNK") & (reference.obs['cluster_id'] != "GLU") & (reference.obs['cluster_id'] != "ASTRO") & (reference.obs['cluster_id'] != "OLIGO") & (reference.obs['cluster_id'] != "GABA")

# Subset reference to only those rows
reference_filtered = reference[valid_idx].copy()

# Check that no NaNs or UNKs remain
print(reference_filtered.obs['cluster_id'].isna().sum())  # Should print 0
print((reference_filtered.obs['cluster_id'] == "UNK").sum())  # Should print 0

# Rename cluster_id to ClusterName as this is the default for tacco
reference_filtered.obs = reference_filtered.obs.rename(columns={'cluster_id': 'ClusterName'})

print(reference_filtered.obs.columns)

# Optionally filter further (e.g. only WT samples)
#reference_filtered = reference_filtered[reference_filtered.obs['group'] == 'WT']


0
0
Index(['group', 'sample_id', 'batch', 'n_genes_by_counts',
       'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts',
       'pct_counts_in_top_20_genes', 'total_counts_mt',
       'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo',
       'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb',
       'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier',
       'ClusterName', '_scvi_batch'],
      dtype='object')


In [5]:
## Read in batch-corrected xenium data
xenium_data_path = "Z:/eroglulab_share/Gabi_Sejourne/Ctnnd2Zbtb20_Manuscript_DevCell_2025/Figure 5 and S4/xenium_dataset/GEO_submission/anndata_objects"
puck = ad.read_h5ad(f'{xenium_data_path}/adata_scvi_xenium_batchcorrected.h5ad')

In [6]:
# Make sure you are using raw counts
np.sum(puck.X)

np.float32(3.2702292e+06)

In [7]:
# matplotlib settings
import matplotlib

highres = False
default_dpi = 100.0 # matplotlib.rcParams['figure.dpi']
if highres:
    matplotlib.rcParams['figure.dpi'] = 648.0
    hr_ext = '_hd'
else:
    matplotlib.rcParams['figure.dpi'] = default_dpi
    hr_ext = ''

axsize = np.array([4,3])*0.5

x = 'SPRING-x'
y = 'SPRING-y'

## Perform label transfer

In [8]:
# run tacco
tc.tl.annotate(puck,reference_filtered,'ClusterName',result_key='ClusterName', multi_center=10, max_annotation = 4, assume_valid_counts=True)

Starting preprocessing
Annotation profiles were not found in `reference.varm["ClusterName"]`. Constructing reference profiles with `tacco.preprocessing.construct_reference_profiles` and default arguments...
Finished preprocessing in 8.85 seconds.
Starting annotation of data with shape (31469, 329) and a reference of shape (95285, 329) using the following wrapped method:
+- maximum annotation: max_annotation=4
   +- platform normalization: platform_iterations=0, gene_keys=ClusterName, normalize_to=adata
      +- multi center: multi_center=10 multi_center_amplitudes=True
         +- bisection boost: bisections=4, bisection_divisor=3
            +- core: method=OT annotation_prior=None
mean,std( rescaling(gene) )  3.3705816346119155 11.338507667016088


C:\Users\Gabrielle\anaconda3\envs\tacco_env\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
C:\Users\Gabrielle\anaconda3\envs\tacco_env\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 4096 or by setting the environment variable OMP_NUM_THREADS=1
  warnings.warn(
C:\Users\Gabrielle\anaconda3\envs\tacco_env\Lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWar

bisection run on 1
bisection run on 0.6666666666666667
bisection run on 0.4444444444444444
bisection run on 0.2962962962962963
bisection run on 0.19753086419753085
bisection run on 0.09876543209876543
Finished annotation in 10.92 seconds.


AnnData object with n_obs × n_vars = 31469 × 331
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'leiden_res0_25', 'leiden_res0_1', 'leiden_res0_5', 'leiden_res0_75', 'condition', 'sample_id', 'leiden_res1_25', 'leiden_res1_0', '_scvi_batch', '_scvi_labels', 'leiden_res0_35', 'cluster_id'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'cluster_id_colors', 'condition_colors', 'dendrogram_leiden_res0_25', 'hvg', 'le

In [9]:
# Convert one-hot/probabilities to single label per row
puck.obs['predicted_type'] = puck.obsm['ClusterName'].idxmax(axis=1)

In [10]:
## Combine Deep and WM into a single cluster

pred = puck.obs['predicted_type']

# convert to a python list for editing
labels = pred.tolist()

# merge two categories manually
labels = ['Deep_WM' if x in ['Deep', 'WM'] else x for x in labels]

# assign back as a new categorical series
puck.obs['predicted_type'] = pd.Categorical(labels)


In [11]:
import pandas as pd
import numpy as np

# Convert Clustername_mc10 column from pandas series to dict for safe writing
if isinstance(puck.uns['ClusterName_mc10'], pd.Series):
    puck.uns['ClusterName_mc10'] = puck.uns['ClusterName_mc10'].to_dict()  # or .to_frame()

# Then try writing again
puck.write(filename=f'{data_path}/adata_tacco_xenium.h5ad')
